# Kedro

Kedro organise un projet de Machine Learning en modules reproductibles.
Ce notebook crée le projet `purchase-predict`, décrit son architecture, puis construit le pipeline `processing`.
Ce pipeline encode le jeu `primary` et le sépare en ensembles d'entraînement et de test.
La version utilisée est Kedro 0.17.0. Une version plus récente peut introduire des incompatibilités.

## Création du projet

La commande suivante crée le projet. Le nom affiché est `purchase-predict`.
Le nom du dépôt et le nom du paquet Python conservent la valeur proposée par défaut (`purchase-predict`, `purchase_predict`).

Architecture générée :

```text
├── conf
│   ├── base
│   │   ├── catalog.yml
│   │   ├── credentials.yml
│   │   ├── logging.yml
│   │   └── parameters.yml
│   ├── local
│   └── README.md
├── data
│   ├── 01_raw
│   ├── 02_intermediate
│   ├── 03_primary
│   ├── 04_feature
│   ├── 05_model_input
│   ├── 06_models
│   ├── 07_model_output
│   └── 08_reporting
├── docs
│   └── source
│       ├── conf.py
│       └── index.rst
├── logs
│   └── journals
├── notebooks
├── pyproject.toml
├── README.md
├── setup.cfg
└── src
    ├── purchase_predict
    │   ├── cli.py
    │   ├── hooks.py
    │   ├── __init__.py
    │   ├── pipelines
    │   │   └── __init__.py
    │   ├── run.py
    │   └── settings.py
    ├── requirements.txt
    ├── setup.py
    └── tests
        ├── __init__.py
        ├── pipelines
        │   └── __init__.py
        └── test_run.py
```

`conf` centralise paramètres, catalogue et secrets.
`data` sépare les jeux selon l'étape du pipeline, des données brutes au modèle sérialisé.
`src` contient le code des pipelines.
`logs` reçoit les journaux Kedro, `docs` la documentation, `notebooks` les notebooks de travail.

Un environnement virtuel à la racine isole les dépendances du projet.

## Catalogue de données

Les chemins de fichiers restent dans `conf/base/catalog.yml`.
Le code désigne un jeu par son nom (`primary`, `dataset`, `X_train`).
Kedro charge alors le fichier avec le type déclaré.

Les dossiers `01_raw`, `02_intermediate` et `03_primary` correspondent aux données brutes, intermédiaires et prêtes à encoder.
`04_feature` reçoit la matrice encodée, `05_model_input` les échantillons d'entraînement et de test, `06_models` les modèles, `07_model_output` et `08_reporting` les sorties.
Cette découpe est la proposition par défaut de Kedro.

Ici, la transformation SQL est faite hors Kedro, avec Spark.
Le fichier reçu est `data/03_primary/primary.csv`.
L'encodage produit `data/04_feature/dataset.csv`, puis les quatre fichiers de `data/05_model_input`.

## Nodes et pipelines

Un node est une tâche : une fonction Python, ses entrées, ses sorties.
Un pipeline est une suite de nodes. Kedro garantit que chaque sortie est disponible pour les nodes suivants.
Le même enchaînement s'applique à chaque exécution.

## Pipeline `processing`

Deux nodes composent ce pipeline.
`encode_features` encode `primary`.
`split_dataset` produit les ensembles d'entraînement et de test. Le paramètre `test_ratio` vient de `conf/base/parameters.yml`.

Copier l'échantillon vers `data/03_primary/primary.csv`.

Créer `src/purchase_predict/pipelines/processing/` avec trois fichiers.
`__init__.py` expose le pipeline.
`nodes.py` définit les fonctions.
`pipeline.py` assemble les nodes.

`encode_features` retourne un dictionnaire : `features` (DataFrame encodé) et `transform_pipeline` (liste des `LabelEncoder`).
Le node mappe ces clés vers les jeux `dataset` et `transform_pipeline`.
`transform_pipeline` n'est pas dans le catalogue : Kedro le conserve en mémoire pour l'exécution.
`split_dataset` retourne `X_train`, `y_train`, `X_test` et `y_test`.

Installer les dépendances dans l'environnement virtuel, puis exécuter le pipeline.
Le nom `processing` est celui sous lequel le pipeline est enregistré dans le projet.
Les quatre CSV apparaissent dans `data/05_model_input`.

## Visualisation

`kedro-viz` affiche le graphe des pipelines dans le navigateur.
L'environnement virtuel doit être actif.
Une fois le serveur lancé, l'interface est accessible via `jupyter://user-redirect/proxy/4141/`.